In [2]:
pip install folium pandas seaborn matplotlib



   ---------------------------------------- 0/3 [xyzservices]
   ------------- -------------------------- 1/3 [branca]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import folium
import seaborn as sns
import matplotlib.pyplot as plt

print(pd.__version__)


3.0.1


In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [9]:
import pandas as pd

URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"

# 直接讀取 CSV
spacex_df = pd.read_csv(URL)

# 檢查前幾行
print(spacex_df.head())


   Flight Number        Date Time (UTC) Booster Version  Launch Site  \
0              1  2010-06-04   18:45:00  F9 v1.0  B0003  CCAFS LC-40   
1              2  2010-12-08   15:43:00  F9 v1.0  B0004  CCAFS LC-40   
2              3  2012-05-22    7:44:00  F9 v1.0  B0005  CCAFS LC-40   
3              4  2012-10-08    0:35:00  F9 v1.0  B0006  CCAFS LC-40   
4              5  2013-03-01   15:10:00  F9 v1.0  B0007  CCAFS LC-40   

                                             Payload  Payload Mass (kg)  \
0               Dragon Spacecraft Qualification Unit                0.0   
1  Dragon demo flight C1, two CubeSats,  barrel o...                0.0   
2                             Dragon demo flight C2+              525.0   
3                                       SpaceX CRS-1              500.0   
4                                       SpaceX CRS-2              677.0   

       Orbit         Customer        Landing Outcome  class        Lat  \
0        LEO           SpaceX  Failure   (

In [10]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [11]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [12]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [13]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label


In [14]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [15]:
marker_cluster = MarkerCluster()


In [17]:
print(spacex_df.columns)


Index(['Launch Site', 'Lat', 'Long', 'class'], dtype='str')


In [19]:
# 根據 Class 值建立 marker_color column
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# 檢查前幾行
spacex_df[['Launch Site', 'class', 'marker_color']].head()


,Launch Site,class,marker_color
0,CCAFS LC-40,0,red
1,CCAFS LC-40,0,red
2,CCAFS LC-40,0,red
3,CCAFS LC-40,0,red
4,CCAFS LC-40,0,red


In [22]:
import folium

# 建立地圖
launch_map = folium.Map(location=[spacex_df["Lat"].mean(),
                                  spacex_df["Long"].mean()],
                        zoom_start=4)

# 加 CircleMarker
for i, row in spacex_df.iterrows():
    folium.CircleMarker(
        location=[row["Lat"], row["Long"]],
        radius=5,  # marker 大小，可以改成 row["PayloadMass"]/1000 做比例
        color=row["marker_color"],
        fill=True,
        fill_color=row["marker_color"],
        popup=f"{row['Launch Site']} | Class: {row['class']}"
    ).add_to(launch_map)

launch_map


In [25]:
import folium
from folium.plugins import MarkerCluster

# 建立地圖
launch_map = folium.Map(location=[spacex_df["Lat"].mean(),
                                  spacex_df["Long"].mean()],
                        zoom_start=4)

# 建立 MarkerCluster
marker_cluster = MarkerCluster().add_to(launch_map)

# 加 marker 入 cluster
for i, row in spacex_df.iterrows():
    folium.Marker(
        location=[row["Lat"], row["Long"]],
        popup=f"{row['Launch Site']} | Class: {row['class']}",
        icon=folium.Icon(color=row["marker_color"], icon="rocket")
    ).add_to(marker_cluster)

launch_map


In [27]:
import folium
from folium.plugins import MousePosition

# 建立地圖
launch_map = folium.Map(location=[spacex_df["Lat"].mean(),
                                  spacex_df["Long"].mean()],
                        zoom_start=4)

# 加 MarkerCluster (之前做過)
from folium.plugins import MarkerCluster
marker_cluster = MarkerCluster().add_to(launch_map)

for i, row in spacex_df.iterrows():
    folium.Marker(
        location=[row["Lat"], row["Long"]],
        popup=f"{row['Launch Site']} | Class: {row['class']}",
        icon=folium.Icon(color=row["marker_color"], icon="rocket")
    ).add_to(marker_cluster)

# 加 MousePosition plugin
MousePosition().add_to(launch_map)

launch_map


In [28]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [29]:
# Launch Site 座標
launch_lat, launch_lon = 28.5623, -80.5774

# Coastline 座標（你用 MousePosition 記錄）
coast_lat, coast_lon = 28.5630, -80.5700

# 計算距離
distance_km = calculate_distance(launch_lat, launch_lon, coast_lat, coast_lon)
print(f"Distance from Launch Site to Coastline: {distance_km:.2f} km")


Distance from Launch Site to Coastline: 0.73 km


In [31]:
import folium
from folium.features import DivIcon

# Launch Site 座標
launch_site_lat, launch_site_lon = 28.5623, -80.5774
# Coastline 座標
coastline_lat, coastline_lon = 28.56367, -80.57163

# 計算距離
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon,
                                        coastline_lat, coastline_lon)

# 建立地圖
site_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=14)

# Launch Site marker
folium.Marker(
    [launch_site_lat, launch_site_lon],
    popup="Launch Site",
    icon=folium.Icon(color="blue", icon="rocket")
).add_to(site_map)

# Coastline marker
folium.Marker(
    [coastline_lat, coastline_lon],
    popup="Coastline",
    icon=folium.Icon(color="green", icon="info-sign")
).add_to(site_map)

# 距離標籤
folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(150, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:.2f} KM</b></div>'.format(distance_coastline),
    )
).add_to(site_map)

# PolyLine 連接 Launch Site → Coastline
coordinates = [(launch_site_lat, launch_site_lon), (coastline_lat, coastline_lon)]
lines = folium.PolyLine(locations=coordinates, weight=2, color="red")
site_map.add_child(lines)

site_map


In [32]:
import folium
from folium.features import DivIcon

# Launch Site 座標
launch_site_lat, launch_site_lon = 28.5623, -80.5774

# 其他點座標
city_lat, city_lon = 28.5383, -81.3792
rail_lat, rail_lon = 28.5640, -80.5700
highway_lat, highway_lon = 28.5650, -80.5720

# 建立地圖
site_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=10)

# Launch Site marker
folium.Marker([launch_site_lat, launch_site_lon],
              popup="Launch Site",
              icon=folium.Icon(color="blue", icon="rocket")).add_to(site_map)

# Closest City marker + PolyLine
folium.Marker([city_lat, city_lon],
              popup="Closest City",
              icon=folium.Icon(color="purple", icon="info-sign")).add_to(site_map)
folium.PolyLine([(launch_site_lat, launch_site_lon), (city_lat, city_lon)],
                weight=2, color="purple").add_to(site_map)

# Closest Railway marker + PolyLine
folium.Marker([rail_lat, rail_lon],
              popup="Closest Railway",
              icon=folium.Icon(color="green", icon="train")).add_to(site_map)
folium.PolyLine([(launch_site_lat, launch_site_lon), (rail_lat, rail_lon)],
                weight=2, color="green").add_to(site_map)

# Closest Highway marker + PolyLine
folium.Marker([highway_lat, highway_lon],
              popup="Closest Highway",
              icon=folium.Icon(color="red", icon="road")).add_to(site_map)
folium.PolyLine([(launch_site_lat, launch_site_lon), (highway_lat, highway_lon)],
                weight=2, color="red").add_to(site_map)

site_map
